<a href="https://colab.research.google.com/github/AI-KBhutta/tp53-sequence-aligner/blob/main/tp53_seq_aligner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
seq1 = "EVQLVESGGGLVQPGGSLRLS" #Healthy B-cell framework
seq2 = "EVQLVQSGGGLVHPGGSLRLS" #Cancer/Lymphoma Variant

In [2]:
MATCH = 1
MISMATCH = -1
GAP_PENALTY = -2
print("Sequences and scoring rules successfully loaded!")

Sequences and scoring rules successfully loaded!


In [3]:
rows = len(seq1) + 1 #Vertically down
cols = len(seq2) + 1 #Horizontally across

matrix = [[0 for _ in range(cols)] for _ in range(rows)]
print(f"Grid successfully created! It is {rows} rows tall by {cols} columns wide.")

Grid successfully created! It is 22 rows tall by 22 columns wide.


In [4]:
for i in range(1, rows):
  matrix[i][0] = matrix[i-1][0] + GAP_PENALTY

for j in range(1, cols):
  matrix[0][j] = matrix[0][j-1] + GAP_PENALTY

import pandas as pd
print("A snapshot of the top-left corner of your scoring map:")
print(pd.DataFrame(matrix).iloc[0:5, 0:5])

A snapshot of the top-left corner of your scoring map:
   0  1  2  3  4
0  0 -2 -4 -6 -8
1 -2  0  0  0  0
2 -4  0  0  0  0
3 -6  0  0  0  0
4 -8  0  0  0  0


In [5]:
for i in range(1, rows):
  for j in range(1, cols):

    if seq1[i-1] == seq2[j-1]:
      diagonal_score = matrix[i-1][j-1] + MATCH
    else:
      diagonal_score = matrix[i-1][j-1] + MISMATCH

    up_score = matrix[i-1][j] + GAP_PENALTY

    left_score = matrix[i][j-1] + GAP_PENALTY

    matrix[i][j] = max(diagonal_score, up_score, left_score)

print("The scoring matrix is completely filled!")

The scoring matrix is completely filled!


In [6]:
align1 = ""
align2 = ""

i = len(seq1)
j = len(seq2)

while i > 0 or j > 0:
  if i > 0 and j > 0 and (matrix[i][j] == matrix [i-1][j-1] + (MATCH if seq1[i-1] == seq2[j-1] else MISMATCH)):
    align1 += seq1[i-1]
    align2 += seq2[j-1]
    i -= 1
    j -= 1
  elif i > 0 and (matrix[i][j] == matrix[i-1][j] + GAP_PENALTY):
    align1 += seq1[i-1]
    align2 += "-"
    i -= 1
  else:
    align1 += "-"
    align2 += seq2[j-1]
    j -= 1

print("Traceback complete! The computer found the optimal path.")


Traceback complete! The computer found the optimal path.


In [7]:
align1 = align1[::-1]
align2 = align2[::-1]

print("FINAL SEQUENCE ALIGNMENT:")
print("-------------------------")
print(f"Healthy BCR:  {align1}")
print(f"Lymphoma BCR: {align2}")

FINAL SEQUENCE ALIGNMENT:
-------------------------
Healthy BCR:  EVQLVESGGGLVQPGGSLRLS
Lymphoma BCR: EVQLVQSGGGLVHPGGSLRLS


In [8]:
match_line = ""
mutations_found = 0
mutation_details = []

for position in range(len(align1)):
  if align1[position] == align2[position]:
    match_line += "|"

  elif align1[position] == "-" or align2[position] == "-":
    match_line += " "
    mutations_found += 1
    mutation_details.append(f"Position {position + 1}: Gap detected")

  else:
    match_line += "*"
    mutations_found += 1
    mutation_details.append(f"Position {position + 1}: '{align1[position]}' mutated to '{align2[position]}'")

# Calculate similarity after the loop finishes
number_of_matches = len(align1) - mutations_found
# Ensure len(align1) is not zero to avoid division by zero
similarity = (number_of_matches / len(align1)) * 100 if len(align1) > 0 else 0

print("ADVANCED ALIGNMENT ANALYSIS...")
print("------------------------------")
print(f"Healthy: {align1}")
print(f"         {match_line}")
print(f"Cancer:  {align2}")
print("------------------------------")
print(f"Sequence Identity: {similarity:.1f}%")
print(f"Total Mutations: {mutations_found}")
for detail in mutation_details:
  print(f" - {detail}")


ADVANCED ALIGNMENT ANALYSIS...
------------------------------
Healthy: EVQLVESGGGLVQPGGSLRLS
         |||||*||||||*||||||||
Cancer:  EVQLVQSGGGLVHPGGSLRLS
------------------------------
Sequence Identity: 90.5%
Total Mutations: 2
 - Position 6: 'E' mutated to 'Q'
 - Position 13: 'Q' mutated to 'H'


In [9]:
#Reusable machine defined
#It takes 2 inputs: the healthy sequence and the mutated sequence.
def analyze_mutation(healthy_seq, cancer_seq, patient_id):
#--Matrix Setup--
  MATCH = 1; MISMATCH = -1; GAP_PENALTY = -2
  rows = len(healthy_seq) + 1
  cols = len(cancer_seq) + 1
  matrix = [[0 for _ in range(cols)] for _ in range(rows)]

  for i in range(1, rows): matrix[i][0] = matrix[i-1][0] + GAP_PENALTY
  for j in range(1, cols): matrix[0][j] = matrix[0][j-1] + GAP_PENALTY
#--Scoring--
  for i in range(1, rows):
    for j in range(1, cols):
      diagonal = matrix[i-1][j-1] + (MATCH if healthy_seq[i-1] == cancer_seq[j-1] else MISMATCH)
      up = matrix[i-1][j] + GAP_PENALTY
      left = matrix[i][j-1] + GAP_PENALTY
      matrix[i][j] = max(diagonal, up, left)
  #--Traceback--
  align1 = ""; align2 = ""
  i = len(healthy_seq); j= len(cancer_seq)

  while i > 0 or j > 0:
    if i > 0 and j > 0 and (matrix[i][j] == matrix[i-1][j-1] + (MATCH if healthy_seq[i-1] == cancer_seq[j-1] else MISMATCH)):
     align1 += healthy_seq[i-1]; align2 += cancer_seq[j-1]
     i -= 1; j -= 1
    elif i > 0 and (matrix[i][j] == matrix[i-1][j] + GAP_PENALTY):
      align1 += healthy_seq[i-1]; align2 += "-"
      i -= 1
    else:
      align1 += "-"; align2 += cancer_seq[j-1]
      j -= 1
  align1 = align1[:: -1]; align2 = align2[:: -1]

  #--Analysis--
  match_line = ""; mutations_found =  0
  for pos in range(len(align1)):
    if align1[pos] == align2[pos]: match_line += "|"
    elif align1[pos] == "-" or align2[pos] == "-": match_line += " "; mutations_found += 1
    else: match_line += "*"; mutations_found += 1

  number_of_matches = len(align1) - mutations_found
  similarity = (number_of_matches / len(align1)) * 100 if len(align1) > 0 else 0

#--Output--
  print(f"\n RESULTS FOR {patient_id}")
  print(f"Healthy: {align1}")
  print(f"         {match_line}")
  print(f"Tumor:   {align2}")
  print(f"Sequence Identity: {similarity: .1f}% | Mutations: {mutations_found}")
  print("-" *35)

# Batch Processing: Testing multiple patients

#Wildtype (healthy) baseline TP53 sequence
wt_p53 = "SQKTYQGSYGFRLGFLHSGTAKSVTCTYSPA"

# Array of patient datasets
patient_samples = [
    ("SQKTYQGSYGFRLGFLHSGTAKSVTCTYSPA", "Patient A (No Mutation)"),
    ("SQKTYQGSYGFRLFLHSGTAKSVTCTHSPA", "Patient B (Aggressive Lymphoma)"),
    ("SQKTYQGSYGFRLGFLHSGTAKSVTCTYSTA", "Patient C (Mild Variant)")

]

print("INITIALIZING TP53 BATCH SCREENING...")
for sequence, name in patient_samples:
  analyze_mutation(wt_p53, sequence, name)

INITIALIZING TP53 BATCH SCREENING...

 RESULTS FOR Patient A (No Mutation)
Healthy: SQKTYQGSYGFRLGFLHSGTAKSVTCTYSPA
         |||||||||||||||||||||||||||||||
Tumor:   SQKTYQGSYGFRLGFLHSGTAKSVTCTYSPA
Sequence Identity:  100.0% | Mutations: 0
-----------------------------------

 RESULTS FOR Patient B (Aggressive Lymphoma)
Healthy: SQKTYQGSYGFRLGFLHSGTAKSVTCTYSPA
         ||||||||||||| |||||||||||||*|||
Tumor:   SQKTYQGSYGFRL-FLHSGTAKSVTCTHSPA
Sequence Identity:  93.5% | Mutations: 2
-----------------------------------

 RESULTS FOR Patient C (Mild Variant)
Healthy: SQKTYQGSYGFRLGFLHSGTAKSVTCTYSPA
         |||||||||||||||||||||||||||||*|
Tumor:   SQKTYQGSYGFRLGFLHSGTAKSVTCTYSTA
Sequence Identity:  96.8% | Mutations: 1
-----------------------------------
